# 在 Google Colab 上运行 OTTER

在免费的 Colab 运行时里，完整跑一遍真实的 OTTER 流程：

1. **安装** —— 下载已发布的 `otter-install` 二进制并真实运行，包括用 `enva` 创建
   `otter-core` 与 `otter-snakemake` 环境。
2. **模拟参考基因组** —— 构建一个参考 *release*，包含真实的注册表目录结构、manifest
   与 checksum。其中的数据是占位内容；不会下载任何基因组。
3. **用真实 fixture 授权项目** —— 使用仓库内自带的降采样 FASTQ，而不是合成数据，并
   通过 `otter build` 逐个驱动所有场景。
4. **解析 run** —— 对每个 workflow 的**每一个 phase** 执行 `otter run --dry-run`，
   让执行层编译并汇报各 phase 的任务图，而不会真正运行任何一个生信工具。

> **这个 notebook 不做什么。** 它不会比对任何一条 read，也不会执行任何 workflow 任务。
> 参考基因组只是一个 *fixture*：目录结构、manifest、digest 都是真的，序列是占位的。
> `--dry-run` 停在 Craftmake 的 planner。dry-run 只能证明 run 能被解析和规划，不能说明
> 任何科学结果。

运行时：**仅 CPU**。GPU 在这里没有任何收益。

## 0. 环境预检

确认运行时是 Linux/x86-64，并且本 notebook 需要的工具都在。Colab 自带 `git`、`python3`
和 `curl`；OTTER 的发布二进制是静态编译的，因此除了一个很小的辅助程序之外，不需要编译
任何东西。

In [ ]:
import os
import platform
import shutil
import subprocess
import sys
import time


def sh(command, check=True, capture=False, stream=False, echo=None, timeout=None):
    """执行一条 shell 命令，并把命令本身回显出来，让 notebook 读起来像一份操作记录。

    capture=True 会把输出收集起来、在命令结束时一次性打印，适合输出是一整块内容的短命令。
    stream=True 会实时回显输出，多分钟的步骤必须用它：被捕获的长命令在结束前什么都不显示，
    看起来就像卡住了。echo=False 只捕获不打印，用于调用方会自己解析并汇总的输出 —— 一个
    Craftmake plan envelope 是几百 KB 的 JSON，打印出来会把周围的内容全部淹没。
    """
    if echo is not None:
        capture = capture or not stream
    print(f"$ {command}")
    if stream:
        started = time.monotonic()
        process = subprocess.Popen(
            command, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
        )
        for line in process.stdout:
            print(line.rstrip())
        returncode = process.wait(timeout=timeout)
        print(f"  (exit {returncode} after {time.monotonic() - started:.0f}s)")
        if check and returncode != 0:
            raise subprocess.CalledProcessError(returncode, command)
        return subprocess.CompletedProcess(command, returncode)

    result = subprocess.run(
        command,
        shell=True,
        check=check,
        text=True,
        capture_output=capture or echo is False,
        timeout=timeout,
    )
    if capture and echo is not False and result.stdout.strip():
        print(result.stdout.rstrip())
    return result


print("python  :", sys.version.split()[0])
print("kernel  :", platform.system(), platform.release(), platform.machine())
for tool in ("git", "curl", "tar"):
    print(f"{tool:9}:", shutil.which(tool) or "缺少")

assert platform.system() == "Linux", "OTTER 的发布二进制是 Linux 构建。"
assert platform.machine() in ("x86_64", "amd64"), (
    f"本 notebook 使用 amd64 发布产物，无法在 {platform.machine()} 上运行。"
)
print("\n预检通过。")

### 配置

所有可调项集中在这里。默认值的选取目标是让整个 notebook 能无人值守地在大约十五分钟内
跑完。

In [ ]:
from pathlib import Path

# 承载 release 与测试 fixture 的公开仓库。
REPO = "otterlab-bio/otter"
RELEASE_TAG = "latest"          # 可固定为 "v1.1.0" 之类，以获得可复现的运行

# 所有内容的落地位置。/content 是 Colab 会话期间持久化的卷；
# /tmp 由内存支撑，跨 cell 会丢失。
WORK = Path("/content/otter-colab")
INSTALL_DIR = WORK / "bin"       # 存放已发布的二进制
REPO_DIR = WORK / "repo"         # 浅克隆，用于 fixture 与 e2e 脚本
REGISTRY = WORK / "registry"     # 模拟的参考基因组注册表
PROJECTS = WORK / "projects"     # 每个场景一个已授权项目

for directory in (WORK, INSTALL_DIR, REGISTRY, PROJECTS):
    directory.mkdir(parents=True, exist_ok=True)

print("仓库      :", REPO)
print("release   :", RELEASE_TAG)
print("工作目录  :", WORK)

# 各场景授权所用的 fixture。每一个都是仓库中自带的、真实降采样后的测序数据，
# 并与满足它的那个 reference 角色配对。
#
# 这里的 release 标签就是已发布数据集实际使用的标签，因此每一处选择都可以直接换成
# 真实拉取而无需改动标识 —— 见下文「在自己的环境里下载真实参考基因组」。
SCENARIOS = {
    "rrbs": {
        "mode": "RRBS",
        "accession": "SRR31480456",
        "references": {"primary": "hg19@GRCh37.p13-gencode-v19"},
    },
    "rnaseq": {
        "mode": "RNASEQ",
        "accession": "SRR018258",
        "references": {"primary": "hg38@GRCh38-gencode-v44"},
    },
    "bs-pdx": {
        "mode": "RRBS",
        "accession": "SRR36187610",
        "references": {
            "graft": "hg38@GRCh38-gencode-v44",
            "host": "mm10@GRCm38-gencode-M25",
        },
    },
    "rna-pdx": {
        "mode": "RNASEQ",
        "accession": "SRR30880970",
        "references": {
            "graft": "hg38@GRCh38-gencode-v44",
            "host": "mm10@GRCm38-gencode-M25",
        },
    },
}

print("\nscenarios:", ", ".join(SCENARIOS))

## 1. 用真实的 `otter-install` 安装

这里会下载已发布的安装器并运行它。这不是模拟：它会解析 release、拉取全部静态二进制、
部署 Craftmake 的 workflow catalog，并创建 `enva` 环境。

**创建环境是最慢的一步** —— `otter-core` 需要几分钟。那是 `conda` 在求解并下载一整套
真实的生信软件栈，不是本 notebook 的开销。

有两个参数值得了解：

- `-install-dir` 把整个安装放在 Colab 会话目录内，而不是家目录。
- `-non-interactive` 全部采用默认值，这正是本 cell 能无人值守运行的原因。

In [ ]:
installer_url = (
    f"https://github.com/{REPO}/releases/{RELEASE_TAG}/download/"
    "otter-install-linux-amd64-static"
)
installer_path = WORK / "otter-install"

sh(f"curl -fsSL -o {installer_path} {installer_url}")
installer_path.chmod(0o755)
print(f"\n安装器: {installer_path} ({installer_path.stat().st_size:,} bytes)")

### 先看计划，再动手

`-dry-run` 会打印它将执行的每一个动作。先读一遍，之后才能区分是安装器的问题还是网络的
问题。

In [ ]:
# 赋值给变量，而不是让这条调用成为 cell 的最后一个表达式。Jupyter 会把 cell 末尾裸调用的
# 返回值以 repr 形式自动显示，从而把整段捕获到的 stdout 以一行转义文本再打印一遍，
# 把上面可读的输出全部淹没。
plan = sh(
    f"{installer_path} -dry-run -non-interactive "
    f"-releases-repo {REPO} -install-dir {INSTALL_DIR}",
    capture=True,
)

### 执行安装

这里**没有**传 `-skip-envs`，所以会真正创建环境。如果你想快速跑一遍、只拉取二进制，把
下面的 `SKIP_ENVS` 设为 `True` 即可 —— notebook 其余部分仍然可用，因为授权与规划步骤
并不需要环境已经存在。

In [ ]:
SKIP_ENVS = False  # 设为 True 可跳过数分钟的环境创建

skip_flag = "-skip-envs" if SKIP_ENVS else ""

# 用 stream=True 而不是 capture=True：这一步会下载一整套生信软件栈，可能耗时数分钟。
# 用 capture 会在结束前什么都不显示，看起来就像卡住了。赋值而不是裸调用，
# 以免 Jupyter 再额外显示一次它的 repr。
install = sh(
    f"{installer_path} -non-interactive {skip_flag} "
    f"-releases-repo {REPO} -install-dir {INSTALL_DIR}",
    stream=True,
)

### 检查装了什么

把安装好的工具和环境都打印出来，这样「装了一半」会在这里就暴露，而不是在三个 cell 之后
变成一个难以定位的报错。

In [ ]:
os.environ["PATH"] = f"{INSTALL_DIR}:{os.environ['PATH']}"

print("=== 已安装的二进制 ===")
for binary in sorted(INSTALL_DIR.iterdir()):
    if binary.is_file() and os.access(binary, os.X_OK):
        print(f"  {binary.name}")

print("\n=== otter 与 craftmake 版本 ===")
sh("otter --version", capture=True)
sh("craftmake --version", capture=True)

print("=== 安装器部署的 workflow catalog ===")
catalog = INSTALL_DIR / "workflows"
print(f"  {catalog}: {sorted(p.name for p in catalog.iterdir()) if catalog.is_dir() else 'MISSING'}")

In [ ]:
print("=== enva 环境 ===")
if SKIP_ENVS:
    print("  已跳过（-skip-envs）")
else:
    sh("enva list", capture=True, check=False)

    # 证明 otter-core 不只是「被列出」，而是真的可用：这正是
    # 「环境目录存在」与「环境能用」的区别。
    print("\n=== otter-core 真的能运行工具吗？ ===")
    result = sh("enva run otter-core -- fastqc --version", capture=True, check=False)
    print("  可用:", result.returncode == 0)

## 2. 获取 fixture 与演练辅助程序

有两样东西必须来自仓库而不是 release：

- `testdata/` 下的**降采样 FASTQ fixture**，各场景就是用它授权的；
- **`stub-registry`**，一个小 Go 程序，它按生产环境的目录结构写出一个参考 release，然后
  用**生产环境**的校验器去验证它。

只有这一个辅助程序需要编译，所以这个 cell 会在缺少 Go 时安装它。浅克隆加上一次 Go 编译，
通常远不到一分钟。

In [ ]:
if not (REPO_DIR / ".git").is_dir():
    # 浅克隆且不带 submodule：这次克隆只为了 fixture 和演练脚本，
    # 工具链来自 release。
    sh(
        f"git clone --depth 1 --no-recurse-submodules "
        f"https://github.com/{REPO}.git {REPO_DIR}",
        stream=True,
    )
else:
    print(f"复用已有克隆：{REPO_DIR}")

print()
sh(f"git -C {REPO_DIR} log --oneline -1", capture=True)

fixture_root = REPO_DIR / "testdata/gate6/craftmake-downsample-20260906/fastq"
print(f"\nfixture: {fixture_root}")
print("  样本编号:", sorted(p.name for p in fixture_root.iterdir()))

In [ ]:
# stub-registry 是本 notebook 唯一需要编译的程序。
if shutil.which("go") is None:
    print("未找到 Go，正在安装（几秒钟）。")
    sh("apt-get -qq update && apt-get -qq install -y golang-go", check=False)

go_path = shutil.which("go")
print("go:", go_path or "仍然缺少")
if go_path:
    # 赋值而不是裸调用，以免 Jupyter 再额外显示一次它的 repr。
    go_version = sh("go version", capture=True)

In [ ]:
stub_registry = INSTALL_DIR / "stub-registry"

if go_path is None:
    raise RuntimeError(
        "stub-registry 需要 Go 工具链。请安装 Go，或预先编译好该二进制并放到 "
        f"{stub_registry}。"
    )

sh(f"cd {REPO_DIR} && go build -o {stub_registry} ./internal/e2esupport/cmd/stub-registry")
print(f"\nstub-registry: {stub_registry.stat().st_size:,} bytes")

## 3. 模拟参考基因组

规范项目并不指向某个 FASTA 路径。它锁定的是一个**注册表 release**：一个逻辑 id、一个
release 标签，以及该 release 内全部内容的 manifest digest。所以要满足一个项目，最快的
办法就是写出一个 release。

`stub-registry` 正是这么做的，并且随后会用**生产环境**的参考基因组校验器检查自己的输出。
你得到的是一个真实的注册表结构 —— `reference.yaml`、`manifest.json`、
`checksums.sha256`、`fasta/`、`annotations/`、`indexes/` —— 只是其中的数据是占位的。

这个区别是关键：**身份校验机制是真的，序列不是。** 因此本 notebook 对 run 所做的一切
断言，都只关于解析与规划，与生物学无关。

### 为什么这里模拟而不是下载

**存储空间。** 一个真实 release 是数十 GB 级别。仅人类 hg19 一个 release 就同时包含
Bisulfite Genome、Bowtie2 索引和 STAR 索引 —— 对 GRCh38 来说，单是 STAR 索引就有约
30 GB。免费的 Colab 运行时大约只有 100 GB 磁盘，而且是与环境创建、workflow 自身的中间
文件共享的临时空间，因此下载一两个 release 就会占掉大部分，留给流程中间产物的空间就不
够了。

另外两个原因：速度 —— 下载会和 `enva` 抢占网络；以及诚实 —— 一个先花十分钟下载才让你
看到东西的教程，作为教学工具不如几秒钟就进入正题的那个。

### 在自己的环境里下载真实参考基因组

当你磁盘足够时，OTTER 通过安装器来安装参考 release。已发布的数据集是公开的，包含五个
release：

```text
fallingstar10/xdxtools-genomes
  genomes/hg19/GRCh37.p13-gencode-v19
  genomes/hg38/GRCh38-gencode-v44
  genomes/mm10/GRCm38-gencode-M25
  genomes/mm39/GRCm39-gencode-vM39
  genomes/mm9/NCBIM37-gencode-M1
```

**方案 A —— 拉取已发布的 release（推荐）。** 这会下载预编译好的索引，从而完全跳过索引
构建：

```bash
# 要拉取哪些 release；以逗号分隔的 <id>@<release> 列表。
export OTTER_REFERENCE_FETCH_RELEASES="hg19@GRCh37.p13-gencode-v19"

# 注册表落地位置。注意：fetch 读的是它自己的变量。工具链其余部分使用的
# OTTER_REFERENCE_ROOT 在这里**不会**被读取 —— 不设这个变量会写到
# ~/.otter/references，那通常不是你想要的位置。
export OTTER_REFERENCE_FETCH_REGISTRY_ROOT=/shared/references

otter-install -reference-fetch -non-interactive
```

只拉取部分内容，可以加 `OTTER_REFERENCE_FETCH_ASSETS`，例如
`bismark,bowtie2,fasta,annotations` 用来跳过你并不需要的 30 GB STAR 索引。如果
huggingface.co 被墙，可以用 `OTTER_REFERENCE_FETCH_BASE_URL` 指向镜像。

**方案 B —— 本地构建一个 release。** 当数据集里没有你要的物种时用这条路。你提供 FASTA
和 GTF，工具链会依次运行 `samtools faidx`、`bismark_genome_preparation`、
`bowtie2-build` 和 `STAR --runMode genomeGenerate`，然后发布一个不可变的 release：

```bash
otter reference build \
  --id myspecies --release ASM123-ensembl-110 \
  --organism 'Mus musculus' --assembly ASM123 \
  --fasta /staging/asm123.fa.gz --gtf /staging/asm123.gtf.gz
```

这是慢路径 —— 哺乳动物基因组的 STAR 索引可能要跑几个小时 —— 但新参考基因组进入注册表
正是走这条路。

**方案 C —— 直接从网页下载。** 该数据集是一个 Hugging Face dataset 仓库，网页界面可以
直接用，而每个 release 都对应注册表的结构：

```text
https://huggingface.co/datasets/fallingstar10/xdxtools-genomes
```

下载你需要的那个 release 的压缩包，解压到
`<registry-root>/genomes/<id>/<release>/`，保持压缩包内已有的目录名即可。

### 然后就像本 notebook 一样使用它

无论走哪条路，walkthrough 的其余部分都不变：`otter create` 或 `otter build` 会向注册表
校验该 release 并锁定其 manifest digest，`otter config resolve` 则把校验过的 release
路径写进 run snapshot。把 `--reference-root` 指向你的注册表，并写出 release 名即可：

```bash
otter build --project-root my_project \
  --fastq ./fastq --pdata ./samples.csv --mode RRBS \
  --reference-root /shared/references \
  --reference-primary hg19@GRCh37.p13-gencode-v19 \
  --backend local
```

你的项目锁定的 digest，与本 notebook 中模拟 release 所携带的 digest 走的是同一套机制
—— 身份校验并不关心数据是否真实。

In [ ]:
# 各场景会选用的全部参考基因组。PDX 场景会同时声明 graft 与 host，
# 因此两个物种都需要。
#
# 这些必须与上文 SCENARIOS 中的选择一致：场景只能选用注册表中确实存在的
# release，所以这两个列表是同一份契约。这里的标签就是已发布数据集实际使用的
# 标签，读者因此可以把模拟注册表换成真实拉取，而无需改动选择。
REFERENCES = [
    {
        "id": "hg19",
        "release": "GRCh37.p13-gencode-v19",
        "organism": "Homo sapiens",
        "assembly": "GRCh37.p13",
        "aliases": "hg19,human,grch37",
    },
    {
        "id": "hg38",
        "release": "GRCh38-gencode-v44",
        "organism": "Homo sapiens",
        "assembly": "GRCh38",
        "aliases": "hg38,human,grch38",
    },
    {
        "id": "mm10",
        "release": "GRCm38-gencode-M25",
        "organism": "Mus musculus",
        "assembly": "GRCm38",
        "aliases": "mm10,mouse,grcm38",
    },
]

for reference in REFERENCES:
    sh(
        f"{stub_registry} --registry-root {REGISTRY} "
        f"--id {reference['id']} --release {reference['release']} "
        f"--organism '{reference['organism']}' --assembly {reference['assembly']} "
        f"--alias {reference['aliases']}"
    )
    print()

In [ ]:
import json

print("=== 注册表目录结构 ===")
sh(f"find {REGISTRY} -maxdepth 4 -mindepth 3 | sort", capture=True)

release_dir = REGISTRY / "genomes/hg19/GRCh37.p13-gencode-v19"
print("=== reference.yaml（release 的身份）===")
print("\n".join(
    f"  {line}" for line in (release_dir / "reference.yaml").read_text().splitlines()[:14]
))

manifest = json.loads((release_dir / "manifest.json").read_text())
print(f"\n=== manifest.json：{len(manifest)} 条记录 ===")
for entry in manifest[:5]:
    print(f"  {entry['path']}")
print(f"  ... 另有 {max(0, len(manifest) - 5)} 条")

# release 契约由 reference.yaml + manifest.json + checksums.sha256 组成。这里只报告
# fixture 实际写出了哪几个，而不是假定三个都在：stub 与生产发布器是两条独立的代码路径，
# 一个断言了某条实现输出的 notebook，会在另一条上直接崩掉。
print("\n=== fixture 写出的契约文件 ===")
contract_files = ["reference.yaml", "manifest.json", "checksums.sha256"]
for name in contract_files:
    path = release_dir / name
    if path.is_file():
        print(f"  {name:<20} {path.stat().st_size:>6} bytes")
    else:
        print(f"  {name:<20} 缺失")

checksums_path = release_dir / "checksums.sha256"
if checksums_path.is_file():
    print("\n=== checksums.sha256 覆盖了 release 身份 ===")
    print("\n".join(
        f"  {line}"
        for line in checksums_path.read_text().splitlines()[:4]
    ))
    print("\n  按其中列出的每个文件逐一校验：")
    # 赋值而不是裸调用：cell 末尾的裸调用会被自动显示为 repr，把 stdout
    # 以一行转义文本再打印一遍。
    checksum_check = sh(f"cd {release_dir} && sha256sum -c checksums.sha256 | tail -3", capture=True)
else:
    print(
        "\nchecksums.sha256 不存在，因此无法在这里演示 sha256sum -c。"
        "\n该文件是 release 契约的一部分，它缺失说明 fixture 有缺口："
        "\n应当如实报告，而不是把该 release 当作完整的。"
    )

## 4. 用 `otter build` 授权每一个场景

对每个场景，这个 cell 会：

1. 把真实 fixture 按 `otter create` 能识别到的 `*_R1.fastq.gz` / `*_R2.fastq.gz` 命名
   摆放好，并写出与之匹配的 pdata 文件；
2. 运行 **`otter build`** —— 这条命令串联了 `init`、`create`、`config validate` 和
   `config resolve`，然后停在执行之前；
3. 检查 project、samples manifest 和 reference lock 是否确实写出来了。

`build` 刻意只是对同一批函数的快捷封装，而不是第二条授权路径。仓库里的离线演练会断言
两条路径产出的授权产物逐字节相同。

In [ ]:
def stage_inputs(scenario, accession):
    """把一对 fixture 按 `otter create` 期望的命名复制好，并写出对应的 pdata。"""
    project_dir = PROJECTS / scenario
    fastq_dir = project_dir / "fastq"
    fastq_dir.mkdir(parents=True, exist_ok=True)

    source = fixture_root / accession
    shutil.copyfile(source / "R1.fastq.gz", fastq_dir / f"{accession}_R1.fastq.gz")
    shutil.copyfile(source / "R2.fastq.gz", fastq_dir / f"{accession}_R2.fastq.gz")

    pdata = project_dir / "pdata.csv"
    pdata.write_text(
        "sampleid,inline_barcode_sequence,condition\n" f"{accession},,case\n"
    )
    return project_dir, fastq_dir, pdata


def reference_arguments(references):
    """渲染角色参数。primary 是单角色；PDX 则是 graft 加 host。"""
    if "primary" in references:
        return f"--reference-primary {references['primary']}"
    return (
        f"--reference-graft {references['graft']} "
        f"--reference-host {references['host']}"
    )


snapshots = {}

for scenario, spec in SCENARIOS.items():
    print("=" * 72)
    print(f"场景: {scenario}  (mode={spec['mode']}, fixture={spec['accession']})")
    print("=" * 72)

    project_dir, fastq_dir, pdata = stage_inputs(scenario, spec["accession"])
    references = reference_arguments(spec["references"])

    result = sh(
        f"otter build --project-root {project_dir} "
        f"--fastq {fastq_dir} --pdata {pdata} --mode {spec['mode']} "
        f"--jobid {scenario} "
        f"--reference-root {REGISTRY} {references} --backend local",
        capture=True,
        check=False,
    )

    if result.returncode != 0:
        raise RuntimeError(f"otter build failed for {scenario} (exit {result.returncode})")

    # `build` 会把 snapshot 路径单独打印在最后一行。
    snapshot_path = Path(result.stdout.strip().splitlines()[-1].strip())
    assert snapshot_path.name == "run.yaml", f"unexpected snapshot path: {snapshot_path}"
    assert snapshot_path.exists(), f"reported snapshot does not exist: {snapshot_path}"
    snapshots[scenario] = snapshot_path

    for artifact in ("project.yaml", "samples.tsv", "references.lock.yaml"):
        assert (project_dir / artifact).exists(), f"{scenario} is missing {artifact}"

    print(f"  snapshot: {snapshot_path.relative_to(WORK)}")
    print()

print(f"已授权 {len(snapshots)} 个场景: {', '.join(snapshots)}")

### `build` 产出了什么

这个项目目录值得读一遍。注意 Snakemake 规则被固定在 `workflows/rules/`，也就是与
Snakefile 放在一起，因为 Snakefile 的 `include:` 指令是相对它自身解析的。项目根目录里
不会写入任何东西。

In [ ]:
sample_project = PROJECTS / "rrbs"

print("=== 项目目录结构 ===")
sh(f"find {sample_project} -maxdepth 2 -type d | sort", capture=True)

print("=== project.yaml ===")
print("\n".join(
    f"  {line}" for line in (sample_project / "project.yaml").read_text().splitlines()
))

print("\n=== samples.tsv（路径相对项目根目录）===")
print("\n".join(
    f"  {line}" for line in (sample_project / "samples.tsv").read_text().splitlines()
))

print("\n=== references.lock.yaml（被锁定的 digest）===")
print("\n".join(
    f"  {line}"
    for line in (sample_project / "references.lock.yaml").read_text().splitlines()
))

print("\n=== 规则固定在 workflows/ 下，而不是项目根目录 ===")
print("  workflows/rules 存在:", (sample_project / "workflows/rules").is_dir())
print("  项目根 rules/ 存在:", (sample_project / "rules").exists())

## 5. 用 `--dry-run` 解析每一个 run

这一步回答的是 *「它真的能解析吗？」*

`--dry-run` 对应 `craftmake plan`：执行层会校验不可变 snapshot、针对安装器部署的 workflow
catalog 编译 DAG，并汇报任务图。它不碰任何输入数据，也不运行任何工具。

每个场景都会断言三件事，所以这个 cell 通过是有具体含义的：

- 进程以 0 退出；
- 返回的是真正的协议 envelope，含 `"command": "plan"` 与 `"ok": true`；
- envelope 非空，也就是它确实规划出了任务，而不只是「成功返回」。

**每个 phase 都会被规划，而不只是 step1。** 一个 run 解析出来的 workflow 是多阶段的，所以
只规划一个 phase 只能证明那一个 phase；后续 phase 消费前面 phase 的产物，解析问题恰恰容易
在那里暴露。

In [ ]:
import json

# 显式指定而不是依赖 PATH：解析器会退回做 PATH 查找，而一个会调整自身环境的
# notebook 不应该因此失败。
craftmake_binary = INSTALL_DIR / "craftmake"
catalog = INSTALL_DIR / "workflows"


def phases_for(workflow):
    """从已部署的 catalog 中按顺序列出某个 workflow 已发布的 phase。

    从 catalog 读取而不是写死：各 workflow 的 phase 集合不同（RNA-seq 没有 step3），
    写死的列表会在新增 phase 的那一刻悄然失效。
    """
    order = {"step1": 0, "step2": 1, "step2-check": 2, "step3": 3, "step3-check": 4, "publish": 5}
    names = [p.stem for p in (catalog / workflow).glob("*.yaml")]
    return sorted(names, key=lambda name: (order.get(name, 99), name))


def plan_phase(scenario, snapshot_path, phase):
    """规划一个 phase 并返回其 envelope。

    用 echo=False，因为一个 plan envelope 是几百 KB 的 JSON：打印它会淹没这个 cell
    真正要产出的汇总。真正有用的是任务数量，它会在下面打印。
    """
    result = sh(
        f"otter run --config {snapshot_path} "
        f"--executor craftmake --phase {phase} "
        f"--dry-run --foreground "
        f"--craftmake-binary {craftmake_binary} --catalog {catalog}",
        echo=False,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(f"dry run failed for {scenario} {phase} (exit {result.returncode})")
    return json.loads(result.stdout)


# 每个 phase 都会被规划，而不只是 step1。一个 run 解析出来的是多阶段 workflow，
# 只规划一个 phase 只能证明那一个 phase；后续 phase 会消费前面 phase 的产物，
# 解析问题恰恰容易在那里暴露。
plan_results = {}
total_tasks = 0

for scenario, snapshot_path in snapshots.items():
    print("=" * 72)
    print(f"规划: {scenario}")
    print("=" * 72)

    # 从 snapshot 自身的 step1 计划里取得 workflow 名，这样 phase 列表来自
    # 这次 run 实际选中的东西，而不是某个命名约定。
    first_envelope = plan_phase(scenario, snapshot_path, "step1")
    workflow = first_envelope["data"]["workflow"]
    phase_names = phases_for(workflow)
    print(f"  workflow: {workflow}")
    print(f"  phases  : {', '.join(phase_names)}")
    print()

    scenario_plans = {}
    for phase in phase_names:
        envelope = (
            first_envelope
            if phase == "step1"
            else plan_phase(scenario, snapshot_path, phase)
        )

        assert envelope.get("command") == "plan", envelope.get("command")
        assert envelope.get("ok") is True, envelope
        tasks = envelope.get("data", {}).get("tasks", [])
        assert tasks, f"{scenario} {phase} returned no tasks"

        scenario_plans[phase] = len(tasks)
        total_tasks += len(tasks)
        print(f"    {phase:<12} {len(tasks):>3} 个任务")
    print()

    plan_results[scenario] = {"workflow": workflow, "phases": scenario_plans}

print(f"已规划 {len(plan_results)} 个场景的全部 phase，共 {total_tasks} 个任务")
print("没有执行任何任务")

## 6. 汇总

一张紧凑的表，汇总装了什么、授权了什么、规划了什么。

In [ ]:
import unicodedata
from datetime import datetime, timezone

print("=" * 72)
print("OTTER Colab 演练 —— 汇总")
print("=" * 72)
print(f"完成时间  : {datetime.now(timezone.utc):%Y-%m-%d %H:%M:%S} UTC")
print(f"仓库      : {REPO}")
print(f"release   : {RELEASE_TAG}")
print(f"环境      : {'已跳过' if SKIP_ENVS else 'otter-core（已创建）'}")
print()

# CJK glyphs are two columns wide, so pad by display width instead of by character count
# or the columns will not line up.
def pad(text, width, right=False):
    filler = " " * max(0, width - sum(2 if unicodedata.east_asian_width(ch) in "WF" else 1 for ch in text))
    return filler + text if right else text + filler

header = pad("场景", 10) + pad("mode", 8) + pad("fixture", 12) + pad("workflow", 17) + pad("phase", 6, True) + pad("任务", 6, True)
print(header)
print("-" * len(header))
for scenario, spec in SCENARIOS.items():
    result = plan_results[scenario]
    phase_tasks = result["phases"]
    print(
        pad(scenario, 10) + pad(spec['mode'], 8) + pad(spec['accession'], 12)
        + pad(result['workflow'], 17) + pad(str(len(phase_tasks)), 6, True)
        + pad(str(sum(phase_tasks.values())), 6, True)
    )

print()
print("每个场景都解析成了不可变 snapshot，每个 workflow 的每个 phase 都到达了")
print("Craftmake 的 planner。没有执行任何生信工具，也没有任何结果是科学结论。")
print()
print(f"{WORK} 下的产物：")
print(f"  {INSTALL_DIR}   已发布的二进制")
print(f"  {REGISTRY}      模拟的参考基因组注册表")
print(f"  {PROJECTS}      已授权项目与已解析的 run")

## 接下来可以做什么

**跑完整的离线演练。** 同一个仓库还提供 `scripts/e2e/otter_e2e.sh`，它会驱动 103 个
stage，覆盖全部四个场景、legacy 迁移路径、site profile 分支和 executor 配对契约 —— 其中
包括一个分支：把每个场景授权两遍，逐字节比较 `otter build` 与手工
`init` → `create` → `resolve` 链的产物。

```bash
cd /content/otter-colab/repo
bash scripts/e2e/otter_e2e.sh \
  --otter        /content/otter-colab/bin/otter \
  --craftmake    /content/otter-colab/bin/craftmake \
  --stub-registry /content/otter-colab/bin/stub-registry \
  --installer    /content/otter-colab/otter-install \
  --craftmake-catalog /content/otter-colab/bin/workflows
```

它是离线的：不下载基因组、不构建索引、不执行任何 workflow 任务。

**使用真实的参考基因组。** 用已发布的 release 替换模拟注册表：

```bash
otter-install -reference-fetch \
  -releases-repo otterlab-bio/otter \
  -install-dir /content/otter-colab/bin
```

**读用户手册。** [用户手册](https://github.com/otterlab-bio/otter/blob/main/docs/manual/README.md)
覆盖安装、数据准备、各分析模式、site profile，以及从 legacy 布局迁移。

**常见问题。** 如果提示找不到 `otter`，说明预检 cell 里的 `PATH` 导出被跳过或运行时重启
过了 —— 重新运行那个 cell 即可。如果环境创建失败，notebook 其余部分仍然可用：授权与规划
并不要求环境已经存在。